# Vesuvius Challenge 2025 - V5 Large Patch Training (190×190×190)

## Goal: Beat V4's 0.6027 Val Dice with larger patches

## Key Design (based on V4's success):
1. **SafeInstanceNorm3d** (NOT GroupNorm - V4 proved this works better)
2. **190×190×190 patches** (more context than V4's 128³)
3. **Batch size 4** with **accumulation=4** (effective batch=16, same as V4)
4. **LR 0.01** (same as V4, NOT scaled)
5. **Dice + CE + Skeleton** (same as V4, NO clDice - it confused V2)
6. **Soft labeling** (new - to handle noisy labels)

## Speed Optimizations (target: ~5 min/epoch):
- **PATCHES_PER_VOLUME: 4** (not 8) → fewer batches per epoch
- **Lighter augmentation** → no slow spatial transforms on 190³
- **Skeleton iterations: 5** (not 10) → faster loss computation

## Why V4 beat V2:
- V4 (0.6027): SafeInstanceNorm3d, simple loss, batch=16
- V2 (0.5301): GroupNorm, complex loss scheduling, batch=4

## About LB Score:
- Val Dice ≠ LB score (depends on test data distribution)
- V4's 0.6027 val dice could be higher or lower on LB
- Post-processing (threshold, connected components) also affects LB

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & V5 CONFIGURATION - LARGE PATCHES WITH V4's PROVEN SETTINGS
# =============================================================================

import os
import gc
import json
import math
import random
import warnings
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False

warnings.filterwarnings('ignore')

# =============================================================================
# V5 CONFIGURATION - OPTIMIZED FOR SPEED (~5 min/epoch instead of 15)
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v5")
    LOAD_CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v5")
    
    # ==========================================================================
    # PATCH SIZE: 190³ (larger than V4's 128³)
    # ==========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (190, 190, 190)
    
    # Model architecture (SAME as V4 - proven to work)
    FEATURES: List[int] = None  # [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = None  # [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # ==========================================================================
    # TRAINING SETTINGS
    # ==========================================================================
    EPOCHS: int = 300
    BATCH_SIZE: int = 4           # Per-GPU batch size
    ACCUMULATION_STEPS: int = 4   # Effective batch = 16 (same as V4)
    NUM_WORKERS: int = 4
    
    # Learning rate - SAME as V4
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 5.0
    
    # ==========================================================================
    # LOSS WEIGHTS - SAME AS V4
    # ==========================================================================
    DICE_WEIGHT: float = 0.5
    CE_WEIGHT: float = 0.5
    SKELETON_WEIGHT: float = 0.1
    CLDICE_WEIGHT: float = 0.0     # DISABLED
    
    SKELETON_START_EPOCH: int = 0
    SKELETON_WARMUP_EPOCHS: int = 50
    CLDICE_START_EPOCH: int = 9999
    CLDICE_WARMUP_EPOCHS: int = 100
    
    # ==========================================================================
    # LABEL NOISE HANDLING (from Kaggle discussions)
    # ==========================================================================
    USE_SOFT_LABELS: bool = True
    SOFT_LABEL_HIGH: float = 0.95
    SOFT_LABEL_LOW: float = 0.05
    
    # Training control
    RESUME_TRAINING: bool = False
    LOAD_BEST: bool = False
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 10
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    USE_COMPILE: bool = True
    
    # ==========================================================================
    # SPEED OPTIMIZATION: Fewer patches per epoch (was 8, now 4)
    # 786 volumes × 4 patches = 3144 samples → ~786 batches with batch=4
    # ==========================================================================
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 4   # REDUCED from 8 → 2× faster epochs
    PRELOAD_ALL: bool = True
    FG_OVERSAMPLE_RATIO: float = 0.33
    
    # ==========================================================================
    # SPEED OPTIMIZATION: Lighter augmentation for 190³ patches
    # Spatial transforms are SLOW on large volumes - disable them
    # ==========================================================================
    AUG_ROTATION_RANGE: float = 30.0
    AUG_ROTATION_P: float = 0.0    # DISABLED - too slow on 190³
    AUG_SCALE_RANGE: Tuple[float, float] = (0.7, 1.4)
    AUG_SCALE_P: float = 0.0       # DISABLED - too slow on 190³
    
    # Keep fast augmentations
    AUG_GAUSSIAN_NOISE_P: float = 0.1
    AUG_GAUSSIAN_NOISE_STD: Tuple[float, float] = (0.0, 0.1)
    AUG_GAUSSIAN_BLUR_P: float = 0.15
    AUG_GAUSSIAN_BLUR_SIGMA: Tuple[float, float] = (0.5, 1.0)
    AUG_BRIGHTNESS_P: float = 0.15
    AUG_BRIGHTNESS_RANGE: float = 0.3
    AUG_CONTRAST_P: float = 0.15
    AUG_CONTRAST_RANGE: Tuple[float, float] = (0.75, 1.25)
    AUG_GAMMA_P: float = 0.2
    AUG_GAMMA_RANGE: Tuple[float, float] = (0.7, 1.5)
    AUG_GAMMA_INVERT_P: float = 0.05
    AUG_LOW_RES_P: float = 0.0     # DISABLED - slow on 190³
    AUG_LOW_RES_ZOOM: Tuple[float, float] = (0.5, 1.0)
    AUG_MIRROR_P: float = 0.5      # Keep - very fast
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        self.LOAD_CHECKPOINT_DIR = Path(self.LOAD_CHECKPOINT_DIR)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

# Calculate effective batch size
effective_batch = cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS
voxels_per_patch = cfg.PATCH_SIZE[0] * cfg.PATCH_SIZE[1] * cfg.PATCH_SIZE[2]
voxels_128 = 128 * 128 * 128

print("="*70)
print("V5 LARGE PATCH TRAINING - SPEED OPTIMIZED")
print("="*70)
print(f"\n>>> PATCH SIZE: {cfg.PATCH_SIZE} ({voxels_per_patch/voxels_128:.1f}× more voxels than 128³)")
print(f">>> Batch: {cfg.BATCH_SIZE} × {cfg.ACCUMULATION_STEPS} accum = {effective_batch} effective")
print(f">>> Learning rate: {cfg.LEARNING_RATE}")
print(f">>> Epochs: {cfg.EPOCHS}")
print(f">>> torch.compile: {cfg.USE_COMPILE}")
print(f"\n>>> SPEED OPTIMIZATIONS:")
print(f"    PATCHES_PER_VOLUME: {cfg.PATCHES_PER_VOLUME} (was 8)")
print(f"    Rotation/Scale aug: DISABLED (slow on large patches)")
print(f"    Expected epoch time: ~5 min (was 15 min)")
print(f"\n>>> Loss (same as V4):")
print(f"    Dice: {cfg.DICE_WEIGHT}, CE: {cfg.CE_WEIGHT}")
print(f"    Skeleton: {cfg.SKELETON_WEIGHT}")
print(f"\n>>> Soft labels: {cfg.USE_SOFT_LABELS}")
print("="*70)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER (same as V4)
# =============================================================================

class CheckpointManager:
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
        
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
                'skeleton_weight': cfg.SKELETON_WEIGHT,
            }
        }
        
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.load_dir / 'best_model.pth'
            else:
                checkpoint_path = self.load_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        model_to_load = model._orig_mod if hasattr(model, '_orig_mod') else model
        model_to_load.load_state_dict(ckpt['model_state_dict'], strict=True)
        
        if optimizer and 'optimizer_state_dict' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                print("  Optimizer state loaded")
            except:
                print("  Warning: Could not load optimizer state")
        
        if scaler and 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict']:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager defined")

In [ ]:
# =============================================================================
# CELL 3: DATASET WITH AUGMENTATION + SOFT LABELS
# =============================================================================

def load_volume_fast(path: Path) -> np.ndarray:
    return tifffile.imread(str(path))


def augment_spatial(image, label, cfg):
    do_rotation = random.random() < cfg.AUG_ROTATION_P
    do_scale = random.random() < cfg.AUG_SCALE_P
    
    if not do_rotation and not do_scale:
        return image, label
    
    center = np.array(image.shape) / 2.0
    rot_range = cfg.AUG_ROTATION_RANGE * np.pi / 180.0
    
    if do_rotation:
        angle = random.uniform(-rot_range, rot_range)
        axis_pair = random.choice([(0, 1), (0, 2), (1, 2)])
    else:
        angle = 0
        axis_pair = (0, 1)
    
    if do_scale:
        scale = random.uniform(*cfg.AUG_SCALE_RANGE)
    else:
        scale = 1.0
    
    R = np.eye(3)
    c, s = np.cos(angle), np.sin(angle)
    i, j = axis_pair
    R[i, i] = c
    R[i, j] = -s
    R[j, i] = s
    R[j, j] = c
    R = R / scale
    offset = center - R @ center
    
    image_out = ndimage.affine_transform(image, R, offset=offset, order=3, mode='reflect')
    label_out = ndimage.affine_transform(
        label.astype(np.float32), R, offset=offset, order=0, mode='constant', cval=2
    ).astype(np.uint8)
    
    return image_out, label_out


def augment_gaussian_noise(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_NOISE_P:
        return image
    std = random.uniform(*cfg.AUG_GAUSSIAN_NOISE_STD)
    noise = np.random.normal(0, std, image.shape).astype(np.float32)
    return image + noise


def augment_gaussian_blur(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_BLUR_P:
        return image
    sigma = random.uniform(*cfg.AUG_GAUSSIAN_BLUR_SIGMA)
    return gaussian_filter(image, sigma=sigma).astype(np.float32)


def augment_brightness(image, cfg):
    if random.random() >= cfg.AUG_BRIGHTNESS_P:
        return image
    offset = random.uniform(-cfg.AUG_BRIGHTNESS_RANGE, cfg.AUG_BRIGHTNESS_RANGE)
    return image + offset


def augment_contrast(image, cfg):
    if random.random() >= cfg.AUG_CONTRAST_P:
        return image
    factor = random.uniform(*cfg.AUG_CONTRAST_RANGE)
    mean = image.mean()
    return (image - mean) * factor + mean


def augment_gamma(image, cfg):
    if random.random() < cfg.AUG_GAMMA_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = ((image - mn) / rng) ** gamma * rng + mn
    
    if random.random() < cfg.AUG_GAMMA_INVERT_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = rng - ((rng - (image - mn)) / rng) ** gamma * rng + mn
    
    return image


def augment_simulate_low_resolution(image, cfg):
    if random.random() >= cfg.AUG_LOW_RES_P:
        return image
    
    axis = random.randint(0, 2)
    zoom_factor = random.uniform(*cfg.AUG_LOW_RES_ZOOM)
    
    if zoom_factor >= 0.99:
        return image
    
    shape = list(image.shape)
    zoom = [1.0, 1.0, 1.0]
    zoom[axis] = zoom_factor
    
    downsampled = ndimage.zoom(image, zoom, order=0)
    zoom_back = [1.0, 1.0, 1.0]
    zoom_back[axis] = shape[axis] / downsampled.shape[axis]
    upsampled = ndimage.zoom(downsampled, zoom_back, order=3)
    
    if upsampled.shape != tuple(shape):
        result = np.zeros(shape, dtype=np.float32)
        slices = tuple(slice(0, min(s, u)) for s, u in zip(shape, upsampled.shape))
        result[slices] = upsampled[slices]
        return result
    
    return upsampled.astype(np.float32)


def augment_mirror(image, label, cfg):
    for axis in range(3):
        if random.random() < cfg.AUG_MIRROR_P:
            image = np.flip(image, axis)
            label = np.flip(label, axis)
    return np.ascontiguousarray(image), np.ascontiguousarray(label)


def apply_nnunet_augmentations(image, label, cfg):
    image, label = augment_spatial(image, label, cfg)
    image = augment_gaussian_noise(image, cfg)
    image = augment_gaussian_blur(image, cfg)
    image = augment_brightness(image, cfg)
    image = augment_contrast(image, cfg)
    image = augment_gamma(image, cfg)
    image = augment_simulate_low_resolution(image, cfg)
    image, label = augment_mirror(image, label, cfg)
    return image.astype(np.float32), label


class VesuviusDatasetV5(Dataset):
    """
    Dataset optimized for large patches (190×190×190).
    
    Label handling:
    - Value 0: Background
    - Value 1: Foreground (papyrus surface)
    - Value 2: Ignore region (excluded from loss)
    
    Soft labeling (to handle noisy/thick labels):
    - FG (1) → 0.95
    - BG (0) → 0.05
    """
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (190, 190, 190),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            if (self.images_dir / f"{idx}.tif").exists() and \
               (self.labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.volumes = {}
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_fast(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_fast(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                img = (img - img.mean()) / (img.std() + 1e-8)
                self.volumes[vol_id] = (img, lbl)
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if volume is smaller than patch
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        fg = self.fg_coords.get(vol_id)
        force_fg = self.is_train and random.random() < cfg.FG_OVERSAMPLE_RATIO and fg is not None and len(fg) > 0
        
        if force_fg:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(c[0] - pd // 2, d - pd))
            y = max(0, min(c[1] - ph // 2, h - ph))
            x = max(0, min(c[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        if self.is_train:
            img_patch, lbl_patch = apply_nnunet_augmentations(img_patch, lbl_patch, cfg)
        
        # Create masks
        # Label = 1 is foreground, Label = 2 is ignore
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        # ==========================================================================
        # SOFT LABELING: Handle noisy/thick labels
        # ==========================================================================
        # Labels are auto-generated and "thick" - reduce overconfidence
        if cfg.USE_SOFT_LABELS and self.is_train:
            # FG: 1 → 0.95, BG: 0 → 0.05
            fg_mask = fg_mask * (cfg.SOFT_LABEL_HIGH - cfg.SOFT_LABEL_LOW) + cfg.SOFT_LABEL_LOW
            # Keep ignore regions at 0 (they're masked out anyway)
            fg_mask = fg_mask * (1 - ig_mask)
        
        return {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }


print("Dataset with soft labeling ready")

In [ ]:
# =============================================================================
# CELL 4: MODEL WITH SafeInstanceNorm3d (same as V4)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super(SafeInstanceNorm3d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
    
    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))
    
    def forward(self, x):
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("ResEncUNet3D with SafeInstanceNorm3d defined")

In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS (same as V4, but faster skeleton computation)
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        batch_size = pred.shape[0]
        dice_sum = 0.0
        for b in range(batch_size):
            p = pred[b].reshape(-1)
            t = target[b].reshape(-1)
            intersection = (p * t).sum()
            dice_sum += (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
        
        return 1 - dice_sum / batch_size


class CEWithMask(nn.Module):
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=5):
    """Soft skeletonization - REDUCED to 5 iterations for speed on 190³"""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SkeletonRecallLoss(nn.Module):
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2, num_iter: int = 5):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
        self.num_iter = num_iter  # REDUCED from 10 to 5 for speed
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=self.num_iter)
        
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        target_tube = torch.clamp(target_tube, 0, 1)
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        recall = torch.clamp(recall, 0, 1)
        
        return 1 - recall


class CombinedLossV5(nn.Module):
    def __init__(
        self,
        dice_weight: float = 0.5,
        ce_weight: float = 0.5,
        skeleton_weight: float = 0.1,
        skeleton_start: int = 0,
        skeleton_warmup: int = 30,
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.skeleton_weight = skeleton_weight
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        
        raw_weights = [1.0 / (2 ** i) for i in range(4)]
        total = sum(raw_weights)
        self.ds_weights = [w / total for w in raw_weights]
        
        self.dice_loss = DiceLoss()
        self.ce_loss = CEWithMask()
        self.skeleton_loss = SkeletonRecallLoss(num_iter=5)  # REDUCED for speed
    
    def _get_scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        
        dice = self.dice_loss(pred, target, ignore_mask)
        ce = self.ce_loss(pred, target, ignore_mask)
        
        if skel_scale > 0 and self.skeleton_weight > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
            skel = torch.clamp(skel, 0, 2)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        total = (
            self.dice_weight * dice +
            self.ce_weight * ce +
            self.skeleton_weight * skel_scale * skel
        )
        
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            ds_ce = self.ce_loss(ds_out, ds_target, ds_mask)
            ds_loss = self.dice_weight * ds_dice + self.ce_weight * ds_ce
            total = total + self.ds_weights[i] * ds_loss
        
        total = torch.clamp(total, 0, 10)
        
        return {
            'total': total,
            'dice': dice.item(),
            'ce': ce.item(),
            'skeleton': skel.item() if skel_scale > 0 else 0.0,
            'skel_scale': skel_scale,
        }


print("Loss functions defined (skeleton iterations: 5 for speed)")

In [ ]:
# =============================================================================
# CELL 6: TRAINING WITH GRADIENT ACCUMULATION (compatible with torch.compile)
# =============================================================================

def train_one_epoch_accumulated(model, loader, criterion, optimizer, scaler, device, epoch, accumulation_steps):
    """
    Train one epoch with gradient accumulation.
    
    Key: Use no_sync() context for proper gradient accumulation with DDP/compile.
    """
    model.train()
    
    total_loss = 0
    total_dice = 0
    total_ce = 0
    total_skel = 0
    n_batches = 0
    nan_batches = 0
    
    optimizer.zero_grad(set_to_none=True)
    
    n_total = len(loader)
    log_every = max(1, n_total // 10)
    
    pbar = tqdm(total=n_total, desc=f"Epoch {epoch+1}", file=sys.stdout, dynamic_ncols=True)
    
    for batch_idx, batch in enumerate(loader):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        ignore = batch['ignore_mask'].to(device, non_blocking=True)
        
        # Forward pass with AMP
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, labels, ignore, epoch)
            scaled_loss = losses['total'] / accumulation_steps
        
        # Check for NaN/Inf
        if torch.isnan(scaled_loss) or torch.isinf(scaled_loss):
            nan_batches += 1
            if nan_batches <= 3:
                print(f"\n>>> NaN/Inf loss at batch {batch_idx}! Skipping...")
            pbar.update(1)
            continue
        
        # Backward pass
        scaler.scale(scaled_loss).backward()
        
        # Step optimizer every accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == n_total:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            
            if torch.isnan(grad_norm) or torch.isinf(grad_norm):
                nan_batches += 1
                if nan_batches <= 3:
                    print(f"\n>>> NaN gradient at batch {batch_idx}! Skipping step...")
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                pbar.update(1)
                continue
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        total_ce += losses['ce']
        total_skel += losses.get('skeleton', 0)
        n_batches += 1
        
        pbar.update(1)
        pbar.set_postfix({
            'loss': f"{losses['total'].item():.3f}",
            'dice': f"{losses['dice']:.3f}",
        })
        
        # Periodic memory cleanup
        if batch_idx % 100 == 0:
            sys.stdout.flush()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    pbar.close()
    sys.stdout.flush()
    
    if nan_batches > 0:
        print(f"\n>>> WARNING: {nan_batches} NaN batches skipped this epoch!")
    
    if n_batches == 0:
        print("\n>>> CRITICAL: All batches had NaN! Training failed.")
        return None
    
    return {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
        'train_ce_loss': total_ce / n_batches,
        'train_skel_loss': total_skel / n_batches,
        'nan_batches': nan_batches,
    }


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            if isinstance(output, dict):
                output = output['output']
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n_samples += 1
    
    sys.stdout.flush()
    return {'val_dice': total_dice / max(n_samples, 1)}


print(f"Training ready (batch={cfg.BATCH_SIZE} × {cfg.ACCUMULATION_STEPS} = {cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS} effective)")

In [ ]:
# =============================================================================
# CELL 7: MAIN TRAINING LOOP
# =============================================================================

def main_training():
    """V5 Large Patch training - based on V4's success."""
    
    print("="*70)
    print("VESUVIUS V5 LARGE PATCH TRAINING")
    print("="*70)
    print(f"\n>>> Patch size: {cfg.PATCH_SIZE}")
    print(f">>> Batch: {cfg.BATCH_SIZE} × {cfg.ACCUMULATION_STEPS} = {cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS} effective")
    print(f">>> Learning rate: {cfg.LEARNING_RATE}")
    print(f">>> Epochs: {cfg.EPOCHS}")
    print(f">>> Skeleton weight: {cfg.SKELETON_WEIGHT} (warmup: {cfg.SKELETON_WARMUP_EPOCHS})")
    print(f">>> Soft labels: {cfg.USE_SOFT_LABELS}")
    print(f">>> torch.compile: {cfg.USE_COMPILE}")
    print("="*70)
    
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    print("\n[1/4] Loading training data to RAM...")
    train_dataset = VesuviusDatasetV5(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=True,
    )
    
    print("\n[2/4] Loading validation data to RAM...")
    val_dataset = VesuviusDatasetV5(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=0.15,
        patches_per_volume=1,
        preload=True,
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=max(1, cfg.BATCH_SIZE),
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    
    print("\n[3/4] Creating model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    # torch.compile (same as V4)
    if cfg.USE_COMPILE and hasattr(torch, 'compile'):
        print("Compiling model with torch.compile()...")
        # Use default mode for better compatibility with gradient accumulation
        model = torch.compile(model, mode='default')
    
    print(f"Parameters: {count_parameters(model):,}")
    
    criterion = CombinedLossV5(
        dice_weight=cfg.DICE_WEIGHT,
        ce_weight=cfg.CE_WEIGHT,
        skeleton_weight=cfg.SKELETON_WEIGHT,
        skeleton_start=cfg.SKELETON_START_EPOCH,
        skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
    )
    
    # SGD with same settings as V4
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    ckpt_mgr = CheckpointManager(
        save_dir=cfg.CHECKPOINT_DIR,
        load_dir=cfg.LOAD_CHECKPOINT_DIR,
    )
    
    print("\n[4/4] Checking for checkpoint...")
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, None, scaler, load_best=cfg.LOAD_BEST)
    else:
        start_epoch = 0
        print("Starting fresh training from epoch 0")
    
    # Polynomial decay scheduler (same as V4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda e: (1 - e / cfg.EPOCHS) ** 0.9
    )
    if start_epoch > 0:
        for _ in range(start_epoch):
            scheduler.step()
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"Current LR: {current_lr:.6f}")
    print("="*70)
    print("TRAINING STARTED")
    print("="*70)
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        epoch_start = time.time()
        
        train_metrics = train_one_epoch_accumulated(
            model, train_loader, criterion, optimizer, scaler, 
            cfg.DEVICE, epoch, cfg.ACCUMULATION_STEPS
        )
        
        if train_metrics is None:
            print("\n>>> CRITICAL: Training failed due to NaN. Stopping.")
            break
        
        scheduler.step()
        
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        epoch_time = time.time() - epoch_start
        lr = optimizer.param_groups[0]['lr']
        
        log_parts = [
            f"Epoch {epoch+1}/{cfg.EPOCHS}",
            f"{epoch_time:.1f}s",
            f"LR: {lr:.6f}",
            f"Loss: {train_metrics['train_loss']:.4f}",
            f"Dice: {train_metrics['train_dice_loss']:.4f}",
            f"CE: {train_metrics['train_ce_loss']:.4f}",
        ]
        if train_metrics['train_skel_loss'] > 0:
            log_parts.append(f"Skel: {train_metrics['train_skel_loss']:.4f}")
        if val_metrics['val_dice'] > 0:
            log_parts.append(f"Val: {val_metrics['val_dice']:.4f}")
        if train_metrics.get('nan_batches', 0) > 0:
            log_parts.append(f"NaN: {train_metrics['nan_batches']}")
        
        print(" | ".join(log_parts))
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch,
                      {**train_metrics, **val_metrics})
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print("\n" + "="*70)
    print(f"DONE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*70)
    
    return model, ckpt_mgr

print("Ready to train!")

In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================

# =============================================================================
# V5 CONFIGURATION - BASED ON V4's SUCCESS (0.6027 Val Dice)
# =============================================================================
cfg.RESUME_TRAINING = False
cfg.LOAD_BEST = False
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 300
cfg.VALIDATE_EVERY = 5

# =============================================================================
# KEY SETTINGS - MATCH V4's EFFECTIVE CONFIGURATION
# =============================================================================
# V4: batch=16, LR=0.01, SafeInstanceNorm3d → Val Dice 0.6027
# V5: batch=4, accum=4 → effective=16, LR=0.01, SafeInstanceNorm3d

cfg.BATCH_SIZE = 4
cfg.ACCUMULATION_STEPS = 4  # Effective batch = 16 (same as V4)
cfg.LEARNING_RATE = 0.01    # Same as V4

# torch.compile - use 'default' mode for gradient accumulation compatibility
cfg.USE_COMPILE = True

# Skeleton loss - same as V4
cfg.SKELETON_WEIGHT = 0.1
cfg.SKELETON_WARMUP_EPOCHS = 50

# Soft labels - NEW (handles noisy labels from Kaggle discussions)
cfg.USE_SOFT_LABELS = True

effective_batch = cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS

print("\n" + "="*70)
print("V5 LARGE PATCH TRAINING - BASED ON V4's SUCCESS")
print("="*70)
print(f"PATCH_SIZE: {cfg.PATCH_SIZE}")
print(f"BATCH: {cfg.BATCH_SIZE} × {cfg.ACCUMULATION_STEPS} = {effective_batch} (same as V4)")
print(f"LEARNING_RATE: {cfg.LEARNING_RATE} (same as V4)")
print(f"EPOCHS: {cfg.EPOCHS}")
print(f"torch.compile: {cfg.USE_COMPILE}")
print(f"\nLoss (same as V4):")
print(f"  Dice: {cfg.DICE_WEIGHT}")
print(f"  CE: {cfg.CE_WEIGHT}")
print(f"  Skeleton: {cfg.SKELETON_WEIGHT} (warmup {cfg.SKELETON_WARMUP_EPOCHS})")
print(f"  clDice: DISABLED")
print(f"\nLabel noise handling (NEW):")
print(f"  Soft labels: {cfg.USE_SOFT_LABELS}")
print("="*70 + "\n")

# Clear memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# START TRAINING
model, ckpt_mgr = main_training()

In [ ]:
# =============================================================================
# CELL 9: TRAINING STATUS CHECK
# =============================================================================

def check_training_status():
    last_ckpt = cfg.CHECKPOINT_DIR / 'last_checkpoint.pth'
    best_ckpt = cfg.CHECKPOINT_DIR / 'best_model.pth'
    history_file = cfg.CHECKPOINT_DIR / 'history.json'
    
    print("="*70)
    print("V5 LARGE PATCH TRAINING STATUS")
    print("="*70)
    
    if last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location='cpu', weights_only=False)
        print(f"\nLast checkpoint:")
        print(f"  Epoch: {ckpt['epoch'] + 1}/{cfg.EPOCHS}")
        print(f"  Progress: {100 * (ckpt['epoch'] + 1) / cfg.EPOCHS:.1f}%")
        if 'metrics' in ckpt:
            print(f"  Train Loss: {ckpt['metrics'].get('train_loss', 'N/A')}")
            print(f"  Val Dice: {ckpt['metrics'].get('val_dice', 'N/A')}")
    else:
        print("\nNo checkpoint found.")
    
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
        print(f"\nBest model:")
        print(f"  Epoch: {ckpt['best_epoch'] + 1}")
        print(f"  Best Dice: {ckpt['best_score']:.4f}")
    
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        print(f"\nTraining history: {len(history)} epochs recorded")
        if len(history) > 0:
            recent = history[-5:]
            print("\nRecent epochs:")
            for h in recent:
                nan_str = f", NaN: {h.get('nan_batches', 0)}" if h.get('nan_batches', 0) > 0 else ""
                print(f"  Epoch {h['epoch']+1}: loss={h.get('train_loss', 0):.4f}, "
                      f"val_dice={h.get('val_dice', 0):.4f}{nan_str}")
    
    print("="*70)

check_training_status()